# TSNFA Step-by-Step: All Six Algorithms Compared

This notebook walks through the Temporal Spectral Noise-Floor Adaptation (TSNFA) algorithm and five competing methods, visualising what happens at each stage. You will see:

1. How the synthetic noise model generates realistic sensor data
2. How the FFT isolates the event band (Defence 1: Band Selection)
3. How the persistence filter rejects transient spikes (Defence 2)
4. How the adaptive noise floor tracks drift (Defence 3)
5. All six algorithms running on identical data with real event overlays
6. Why each competing method fails, with specific failure mechanisms shown
7. Final scorecard: DR, FP, FN, Precision, and Latency across all methods

**Paper:** Makovetskii & Thomsen, *IEEE Internet of Things Journal*, 2026  
**Repository:** [github.com/Gnacode/IEEE-TSNFA](https://github.com/Gnacode/IEEE-TSNFA)  
**Dashboard:** [gnacode.github.io/IEEE-TSNFA](https://gnacode.github.io/IEEE-TSNFA/)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import deque

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.2

# ===== Simulation parameters (matching the paper) =====
fs = 100; L = 128; Tf = L / fs; P0 = 1.0
A_dB = 6.0; T_cycle = 3600; SNR_dB = 18
emi_amp = 0.3; dig_amp = 2.0; zeta = 6.0
gamma_d = 3; gamma_a = 64; alpha = 1 - 1/gamma_a
bin_lo, bin_hi = 1, 6

COLORS = {'TSNFA':'#2e7d32','Zhang':'#d32f2f','STFT':'#1565c0',
          'DEDaR':'#e65100','SoD':'#6a1b9a','TinyML':'#00838f'}

print(f'Frame: {L} samples at {fs} Hz = {Tf:.2f} s')
print(f'Event band: bins {bin_lo}-{bin_hi} = [{bin_lo*fs/L:.2f}, {bin_hi*fs/L:.2f}] Hz')
print(f'SNR = {SNR_dB} dB = {10**(SNR_dB/20):.1f}x amplitude ratio')
print(f'EMA alpha = {alpha:.4f} (1.6% new, 98.4% old per frame)')

## 1. Signal Generation

The noise model has four components. Noise power P(t) drifts +/-6 dB over a 1-hour sinusoidal cycle.

In [ ]:
def noise_power(t):
    return P0 * 10**((A_dB/10) * np.sin(2 * np.pi * t / T_cycle))

def generate_frame(t_start, include_event=False, event_freq=2.5):
    P = noise_power(t_start)
    n = np.arange(L); t = n / fs
    w_th = np.random.randn(L) * np.sqrt(P)
    w_emi = emi_amp * np.sqrt(P) * np.sin(2*np.pi*60*t + np.random.uniform(0, 2*np.pi))
    w_dig = np.zeros(L)
    if np.random.random() > 0.6:
        bs = np.random.randint(0, max(1, L-20))
        bl = min(np.random.randint(10, 30), L - bs)
        w_dig[bs:bs+bl] = dig_amp*np.sqrt(P)*np.sin(2*np.pi*np.random.uniform(800,2000)*t[bs:bs+bl])
    s = np.zeros(L)
    if include_event:
        s = np.sqrt(P)*10**(SNR_dB/20)*np.exp(-t*0.5)*np.sin(2*np.pi*event_freq*t)
    return s + w_th + w_emi + w_dig, s, P

t_hour = np.linspace(0, 3600, 1000)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3.5))
ax1.plot(t_hour/60, noise_power(t_hour), 'b-', lw=2)
ax1.axhline(P0, color='gray', ls='--', label='$P_0$')
ax1.set(xlabel='Time (min)', ylabel='Noise Power', title='Noise power drift over 1 hour'); ax1.legend()
ax2.plot(t_hour/60, 10*np.log10(noise_power(t_hour)/P0), 'b-', lw=2)
ax2.axhline(0, color='gray', ls='--')
ax2.fill_between(t_hour/60, -6, 6, alpha=0.1, color='blue')
ax2.set(xlabel='Time (min)', ylabel='dB re $P_0$', title='+/-6 dB swing (factor 16 min-to-max)', ylim=(-8,8))
plt.tight_layout(); plt.show()

## 2. What a Single Frame Looks Like

In the time domain, noise-only and noise+event frames look almost identical. In the frequency domain, the event jumps out in bins 1-6.

In [ ]:
np.random.seed(42)
x_n, _, _ = generate_frame(0, include_event=False)
np.random.seed(42)
x_e, s_e, _ = generate_frame(0, include_event=True)

freqs = np.fft.fftfreq(L, 1/fs)[:L//2]
spec_n = np.abs(np.fft.fft(x_n))[:L//2]
spec_e = np.abs(np.fft.fft(x_e))[:L//2]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
t_f = np.arange(L)/fs

axes[0,0].plot(t_f, x_n, 'b-', lw=0.7)
axes[0,0].set(title=f'Noise only (time domain, peak={np.max(np.abs(x_n)):.1f})', ylabel='Amplitude')

axes[0,1].plot(t_f, x_e, 'b-', lw=0.7)
axes[0,1].plot(t_f, s_e, 'g-', lw=2, alpha=0.5, label='Event signal')
axes[0,1].set(title=f'Noise + event (peak={np.max(np.abs(x_e)):.1f})'); axes[0,1].legend(fontsize=9)

for ax, spec, title in [(axes[1,0], spec_n, 'Noise only (spectrum)'), (axes[1,1], spec_e, 'Noise + event')]:
    ax.semilogy(freqs, spec, 'b-', alpha=0.4, lw=0.8)
    ax.bar(freqs[bin_lo:bin_hi+1], spec[bin_lo:bin_hi+1], width=0.5, color='green', alpha=0.7, zorder=5)
    ax.axvspan(0.78, 4.69, alpha=0.06, color='green')
    X_m = np.max(spec[bin_lo:bin_hi+1])
    ax.annotate(f'X(m)={X_m:.0f}', xy=(7, X_m), fontsize=11, color='green', fontweight='bold')
    ax.set(xlabel='Freq (Hz)', ylabel='|X[k]|', title=title, xlim=(0,50))

plt.tight_layout(); plt.show()
print(f'In-band ratio: {np.max(spec_e[bin_lo:bin_hi+1])/np.max(spec_n[bin_lo:bin_hi+1]):.1f}x in frequency domain')
print(f'Time-domain peak ratio: {np.max(np.abs(x_e))/np.max(np.abs(x_n)):.1f}x -- almost invisible')

## 3. Generate Simulation Data (400 frames, 5 events)

We simulate ~8.5 minutes with 5 events placed at different points in the noise cycle. All six algorithms process the identical signal.

In [ ]:
N_FRAMES = 400
EVENT_GROUPS = [list(range(50,54)), list(range(130,134)), list(range(210,214)),
                list(range(300,304)), list(range(370,374))]
EVENT_FRAMES = set()
for g in EVENT_GROUPS: EVENT_FRAMES.update(g)

np.random.seed(123)
frames_x, frames_s, frames_P, frames_t = [], [], [], []
for f in range(N_FRAMES):
    t_now = f * Tf
    x, s, P = generate_frame(t_now, include_event=(f in EVENT_FRAMES))
    frames_x.append(x); frames_s.append(s); frames_P.append(P); frames_t.append(t_now)
frames_t = np.array(frames_t); frames_P = np.array(frames_P)
print(f'{N_FRAMES} frames, {len(EVENT_GROUPS)} events ({len(EVENT_FRAMES)} event frames)')

In [ ]:
# ===== All six algorithm implementations =====

class TSNFA:
    def __init__(self):
        self.N_hat = np.sqrt(P0)*np.sqrt(L); self.buf = deque(maxlen=gamma_d); self.name='TSNFA'
    def process(self, x):
        spec = np.abs(np.fft.fft(x))[:L//2]; X_m = np.max(spec[bin_lo:bin_hi+1])
        self.buf.append(X_m); X_bar = np.mean(self.buf)
        Theta = zeta*self.N_hat; R = X_bar/Theta if Theta>0 else 0; trigger = R>1.0
        if R<0.8: self.N_hat = alpha*self.N_hat + (1-alpha)*X_bar
        return trigger, X_bar, Theta, self.N_hat

class Zhang:
    def __init__(self):
        self.NZ = np.sqrt(P0)*3; self.beta=0.95; self.name='Zhang'
    def process(self, x):
        XZ = np.max(np.abs(x)); Theta = zeta*self.NZ
        R = XZ/Theta if Theta>0 else 0; trigger = R>1.0
        if R<0.8: self.NZ = self.beta*self.NZ + (1-self.beta)*XZ
        return trigger, XZ, Theta, self.NZ

class STFT:
    def __init__(self, cal):
        vals = [np.max(np.abs(np.fft.fft(x))[:L//2][bin_lo:bin_hi+1]) for x in cal]
        self.Theta0 = np.mean(vals)+3*np.std(vals); self.name='STFT'
    def process(self, x):
        X = np.max(np.abs(np.fft.fft(x))[:L//2][bin_lo:bin_hi+1])
        return X>self.Theta0, X, self.Theta0, self.Theta0

class DEDaR:
    def __init__(self):
        self.E_long = L*P0; self.beta_E=0.95; self.name='DEDaR'
    def process(self, x):
        E_s = np.sum(x**2); self.E_long = self.beta_E*self.E_long + (1-self.beta_E)*E_s
        R = E_s/self.E_long if self.E_long>0 else 0
        return R>zeta, E_s, zeta*self.E_long, self.E_long

class SoD:
    def __init__(self, delta=5.0):
        self.x_ref=0.0; self.delta=delta; self.name='SoD'
    def process(self, x):
        trigger = False
        for s in x:
            if abs(s-self.x_ref)>self.delta: self.x_ref=s; trigger=True
        return trigger, np.max(np.abs(x)), self.delta, self.x_ref

class TinyML:
    def __init__(self, cal):
        self.mean_frame = np.mean(cal, axis=0)
        errors = [np.mean((x-np.dot(x,self.mean_frame)/np.dot(self.mean_frame,self.mean_frame)*self.mean_frame)**2) for x in cal]
        self.Theta_ML = np.percentile(errors, 99); self.name='TinyML'
    def process(self, x):
        c = np.dot(x,self.mean_frame)/np.dot(self.mean_frame,self.mean_frame)
        e = np.mean((x-c*self.mean_frame)**2)
        return e>self.Theta_ML, e, self.Theta_ML, self.Theta_ML

print('All six algorithms defined.')

In [ ]:
# ===== Run all algorithms =====
cal_frames = [frames_x[f] for f in range(20) if f not in EVENT_FRAMES]
methods = {'TSNFA':TSNFA(),'Zhang':Zhang(),'STFT':STFT(cal_frames),
           'DEDaR':DEDaR(),'SoD':SoD(delta=5.0),'TinyML':TinyML(cal_frames)}
results = {n:{'triggers':[],'stat':[],'thresh':[],'nf':[]} for n in methods}

for f in range(N_FRAMES):
    for name, alg in methods.items():
        trigger, stat, thresh, nf = alg.process(frames_x[f])
        results[name]['triggers'].append(trigger)
        results[name]['stat'].append(stat)
        results[name]['thresh'].append(thresh)
        results[name]['nf'].append(nf)

for n in results:
    for k in results[n]: results[n][k] = np.array(results[n][k])

# ===== Event association window =====
# The persistence filter (gamma_d=3) means triggers can extend gamma_d-1 = 2 frames
# after the event ends, because the sliding window still contains event energy.
# This is expected behaviour, not a false positive. Standard practice in event
# detection is to associate triggers within a small window of the event.
ASSOC_WINDOW = 2  # frames after event group end
EVENT_ASSOC = set(EVENT_FRAMES)
for g in EVENT_GROUPS:
    for offset in range(1, ASSOC_WINDOW + 1):
        EVENT_ASSOC.add(g[-1] + offset)

# Compute metrics using association window
metrics = {}
for n in methods:
    tr = results[n]['triggers']
    tp = sum(1 for f in range(N_FRAMES) if tr[f] and f in EVENT_ASSOC)
    fp = sum(1 for f in range(N_FRAMES) if tr[f] and f not in EVENT_ASSOC)
    fn = sum(1 for f in range(N_FRAMES) if not tr[f] and f in EVENT_FRAMES)
    dr = 100*tp/max(tp+fn,1); prec = 100*tp/max(tp+fp,1)
    far = fp/(N_FRAMES*Tf/3600)
    metrics[n] = {'TP':tp,'FP':fp,'FN':fn,'DR':dr,'Prec':prec,'FAR':far}

print(f'Event association window: {ASSOC_WINDOW} frames after each event group')
print(f'(accounts for gamma_d persistence filter tail)\n')
print(f'{"Method":<10} {"DR%":>6} {"FP":>6} {"FN":>4} {"Prec%":>7} {"FAR/hr":>8}')
print('-'*48)
for n in ['TSNFA','Zhang','STFT','DEDaR','SoD','TinyML']:
    m = metrics[n]
    print(f'{n:<10} {m["DR"]:>5.1f}% {m["FP"]:>6} {m["FN"]:>4} {m["Prec"]:>6.1f}% {m["FAR"]:>8.1f}')


## 4. Per-Method Visualisation with Event Overlay

Each plot shows:
- **Green shaded bars** = real events (ground truth)
- **Green triangles** = true positives (correctly detected)
- **Red X marks** = false positives (false alarms)
- **Orange down-triangles** = false negatives (missed events)
- **Blue line** = detection statistic
- **Red line** = threshold
- **Orange dashed** = noise floor (where applicable)
- **Bottom panel** = noise power drift in dB for context

**Note on event association:** The persistence filter ($\gamma_d = 3$) means triggers can extend 1-2 frames *after* the event ends, because the sliding window still contains event energy. This is expected behaviour, not a false alarm. Triggers within 2 frames of an event are counted as true positives (standard practice in event detection systems).

In [ ]:
def plot_method(name, explanation):
    r = results[name]; m = metrics[name]; t_min = frames_t/60
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5), sharex=True,
                                     gridspec_kw={'height_ratios': [3, 1]})
    for g in EVENT_GROUPS:
        for ax in [ax1, ax2]:
            ax.axvspan(frames_t[g[0]]/60, frames_t[g[-1]]/60, alpha=0.18, color='green', zorder=0)
    ax1.plot(t_min, r['stat'], color=COLORS[name], lw=0.8, alpha=0.6, label='Detection statistic')
    ax1.plot(t_min, r['thresh'], 'r-', lw=2, alpha=0.8, label='Threshold')
    if name in ['TSNFA','Zhang']:
        ax1.plot(t_min, r['nf'], 'orange', lw=1.5, ls='--', alpha=0.7, label='Noise floor')
    # Use EVENT_ASSOC for TP/FP classification (accounts for persistence tail)
    tp_i = [f for f in range(N_FRAMES) if r['triggers'][f] and f in EVENT_ASSOC]
    fp_i = [f for f in range(N_FRAMES) if r['triggers'][f] and f not in EVENT_ASSOC]
    fn_i = [f for f in range(N_FRAMES) if not r['triggers'][f] and f in EVENT_FRAMES]
    if tp_i: ax1.scatter(frames_t[tp_i]/60, r['stat'][tp_i], color='green', s=80, marker='^', zorder=10, label=f'TP: {len(tp_i)}')
    if fp_i: ax1.scatter(frames_t[fp_i]/60, r['stat'][fp_i], color='red', s=60, marker='x', zorder=10, label=f'FP: {len(fp_i)}', linewidths=2)
    if fn_i: ax1.scatter(frames_t[fn_i]/60, r['stat'][fn_i], color='orange', s=80, marker='v', zorder=10, label=f'FN: {len(fn_i)}')
    ax1.set_ylabel('Magnitude')
    ax1.set_title(f'{name}: DR={m["DR"]:.0f}%, FP={m["FP"]}, FN={m["FN"]}, Prec={m["Prec"]:.1f}%',
                 fontsize=13, fontweight='bold', color=COLORS[name])
    ax1.legend(fontsize=8, loc='upper right', ncol=3)
    ax2.plot(t_min, 10*np.log10(frames_P/P0), 'purple', lw=1.5, alpha=0.7)
    ax2.set(ylabel='Noise (dB)', xlabel='Time (minutes)', ylim=(-8,8))
    ax2.axhline(0, color='gray', ls='--', lw=0.5)
    plt.tight_layout(); plt.show()
    print(f'  {explanation}\n')


### TSNFA (Algorithm 1): All three defences active

In [ ]:
plot_method('TSNFA', 'All three defences active. The adaptive threshold (red) follows the noise floor (orange dashed), rising and falling with the noise power. Events produce clear exceedances. No false triggers.')

### Zhang (Algorithm 3): No spectral band selection

In [ ]:
plot_method('Zhang', 'No FFT: the detection statistic includes 60 Hz EMI and digital noise peaks. The noise floor tracks the composite signal, causing the threshold to fluctuate unpredictably. Events may be missed when the threshold is inflated by EMI, and noise transients breach it when the threshold dips.')

### STFT (Algorithm 4): No adaptive noise floor

In [ ]:
plot_method('STFT', 'Uses the same FFT band selection as TSNFA, so events are clearly visible. But the threshold (red line) was set once during calibration and never updates. When noise power rises above the calibration level, in-band noise exceeds the frozen threshold, causing sustained false triggering during every high-noise phase.')

### DEDaR (Algorithm 5): No band selection, no persistence

In [ ]:
plot_method('DEDaR', 'Computes broadband energy ratio across ALL frequencies. Any energy transient at any frequency (EMI fluctuation, digital burst, wind gust) can push the ratio above 6. Events produce huge ratio spikes and are easily detected, but so does everything else. Catastrophically non-specific.')

### SoD (Algorithm 6): Architecture mismatch

In [ ]:
plot_method('SoD', 'Send-on-Delta is a data-reduction scheme, not a detection algorithm. The reference value random-walks with noise, making events indistinguishable from noise deviations. With delta set high enough to avoid noise triggers, events are also missed. No viable delta exists for this noise environment.')

### TinyML (Algorithm 7): Frozen learned threshold

In [ ]:
plot_method('TinyML', 'The autoencoder was trained on noise at power P0. When noise power drifts, frames look different from training data and reconstruction error rises above the frozen threshold. The same fundamental failure as STFT: a threshold calibrated at one noise level becomes invalid when the noise changes.')

## 5. Trigger Timeline: All Methods at Once

Every trigger from every method on a single timeline. Green bars = real events. Green marks = TP. Red marks = FP.

In [ ]:
order = ['TSNFA','Zhang','STFT','DEDaR','SoD','TinyML']
fig, ax = plt.subplots(figsize=(14, 5))
for g in EVENT_GROUPS:
    ax.axvspan(frames_t[g[0]]/60, frames_t[g[-1]]/60, alpha=0.15, color='green', zorder=0)
for i, n in enumerate(order):
    tr = results[n]['triggers']; trig_f = np.where(tr)[0]
    for f in trig_f:
        c = 'green' if f in EVENT_ASSOC else 'red'  # Use association window
        ax.plot(frames_t[f]/60, i, '|', color=c, markersize=14, markeredgewidth=1.5)
    m = metrics[n]
    ax.text(-0.3, i, f'{n}  (DR={m["DR"]:.0f}%, FP={m["FP"]})', ha='right', va='center',
           fontsize=9, fontweight='bold', color=COLORS[n])
ax.set_yticks(range(len(order))); ax.set_yticklabels(['']*len(order))
ax.set(xlabel='Time (minutes)',
       title='All methods: green bars=events, green|=TP, red|=FP (2-frame association window)')
ax.set_ylim(-0.5, len(order)-0.5); ax.invert_yaxis()
plt.tight_layout(); plt.show()


## 6. Final Scorecard

In [ ]:
names = ['TSNFA','Zhang','STFT','DEDaR','SoD','TinyML']
colors_list = [COLORS[n] for n in names]

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, (title, key) in zip(axes, [('Detection Rate (%)','DR'),('False Positives','FP'),
                                     ('False Negatives','FN'),('Precision (%)','Prec'),('FAR (/hr)','FAR')]):
    vals = [metrics[n][key] for n in names]
    bars = ax.bar(range(len(names)), vals, color=colors_list, alpha=0.85)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    for bar, val in zip(bars, vals):
        if val > 0:
            label = f'{val:.0f}' if val==int(val) or val>10 else f'{val:.1f}'
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), label, ha='center', va='bottom', fontsize=7)
plt.suptitle('Scorecard: All Methods on Identical Signal', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## 7. Defence Composition Table

In [ ]:
print('Defence Composition and Failure Modes')
print('='*95)
print(f'{"Method":<10} {"Band Sel":>10} {"Adaptive NF":>12} {"Persistence":>12} {"DR%":>6} {"FP":>8}  Failure Mode')
print('-'*95)
defences = [
    ('TSNFA',  'YES','YES','YES','All three defences active'),
    ('Zhang',  'NO','YES*','NO','Composite noise inflates threshold'),
    ('STFT',   'YES','NO','NO','Frozen threshold, cannot follow noise drift'),
    ('DEDaR',  'NO','Partial','NO','Broadband energy triggers on any transient'),
    ('SoD',    'NO','NO','NO','Reference random-walks, no viable delta'),
    ('TinyML', 'Partial','NO','NO','Frozen autoencoder threshold, same as STFT'),
]
for n, b, a, p, fail in defences:
    m = metrics[n]
    print(f'{n:<10} {b:>10} {a:>12} {p:>12} {m["DR"]:>5.0f}% {m["FP"]:>8}  {fail}')
print('\nTSNFA is the only method combining all three defences.')
print('Each missing defence produces a specific, predictable failure mode.')

## 8. Generate PDF Report

Run the cell below to generate a downloadable PDF report containing all results,
pseudocode for each algorithm, and the comparison charts. The PDF will be saved
and available for download from Colab.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import textwrap

# ===== Pseudocode definitions (plain text lines) =====
# Each algorithm: title, color, list of (stage_header, [code_lines])
# Code lines: (line_num, code_text, comment)
PSEUDO = {
'TSNFA': {
 'title': 'Algorithm 1: TSNFA Mean Variant',
 'color': '#0D47A1', 'fill': '#E3F2FD',
 'stages': [
  ('Stage 1: Spectral estimation (band selection)', [
   ('1.', 'X  <-  |FFT(x)|', 'time to frequency, magnitude only'),
   ('2.', 'X(m)  <-  max{ X[k] : k in {1,...,6} }', 'select strongest event-band bin'),
  ]),
  ('Stage 2: Digital noise filter (gamma_d persistence)', [
   ('3.', 'append X(m) to buffer B', 'sliding window of gamma_d frames'),
   ('4.', 'if |B| > gamma_d then discard oldest', 'FIFO, keep gamma_d entries'),
   ('5.', 'X_bar(m)  <-  mean(B)', 'X_bar = (1/gamma_d) x sum(B)'),
  ]),
  ('Stage 3: Threshold from last noise-floor update', [
   ('6.', 'Theta(m)  <-  zeta x N_hat(m-1)', 'Theta = 6.0 x previous noise floor'),
  ]),
  ('Stage 4: Detection ratio and trigger', [
   ('7.', 'R(m)  <-  X_bar(m) / Theta(m)', 'ratio of filtered energy to threshold'),
   ('8.', 'if R(m) > 1.0 then E[m] <- 1', 'trigger: in-band > 6x noise floor'),
  ]),
  ('Stage 5: Noise-floor adaptation (gated EMA)', [
   ('9.', 'if R(m) < 0.8 then', 'R_gate=0.8: update only when quiet'),
   ('10.', '    N_hat(m) <- alpha x N_hat(m-1) + (1-alpha) x X_bar(m)', 'alpha=0.984: 1.6% new, 98.4% old'),
   ('', '  else', ''),
   ('', '    N_hat(m) <- N_hat(m-1)', 'freeze: event must not leak in'),
   ('', '  end if', ''),
   ('11.', 'return E[m], N_hat(m)', ''),
  ]),
 ]},
'Zhang': {
 'title': 'Algorithm 3: Time-Domain Adaptive Thresholding (Zhang et al. 2023)',
 'color': '#B71C1C', 'fill': '#FFEBEE',
 'stages': [
  ('Time-domain frame statistic (no FFT, no band selection)', [
   ('1.', 'XZ(m)  <-  max_n |x[n]|', 'peak sample; includes EMI, digital, thermal'),
  ]),
  ('Trigger decision', [
   ('2.', 'R  <-  XZ(m) / NZ(m-1)', 'ratio of peak to noise floor'),
   ('3.', 'if XZ(m) > zeta x NZ(m-1) then E[m] <- 1', 'trigger if peak > 6x noise floor'),
  ]),
  ('Noise-floor update (gated EMA, faster than TSNFA)', [
   ('4.', 'if R < 0.8 then', 'same gate logic as Algorithm 1'),
   ('5.', '    NZ(m) <- beta x NZ(m-1) + (1-beta) x XZ(m)', 'beta=0.95: 5% new, tau~26s'),
   ('', '  else  NZ(m) <- NZ(m-1)', 'freeze during events'),
   ('', '  end if', ''),
   ('6.', 'return E[m]', 'NZ tracks composite, not in-band'),
  ]),
 ]},
'STFT': {
 'title': 'Algorithm 4: Fixed Spectral Mask (Bhoi et al. 2022)',
 'color': '#E65100', 'fill': '#FFF3E0',
 'stages': [
  ('Spectral estimation (identical to Algorithm 1)', [
   ('1.', 'X[k] <- |FFT(x)|  for k in {1,...,6}', 'event band only, rejects EMI'),
   ('2.', 'X_max  <-  max_k{ X[k] }', 'strongest bin magnitude'),
  ]),
  ('Fixed-threshold comparison (set at deployment, never updated)', [
   ('3.', 'if X_max > Theta_0 then E[m] <- 1', 'single frame can trigger'),
   ('4.', 'return E[m]', 'Theta_0 frozen, cannot follow drift'),
  ]),
 ]},
'DEDaR': {
 'title': 'Algorithm 5: Energy-Ratio Triggering (Hussein et al. 2022)',
 'color': '#4A148C', 'fill': '#F3E5F5',
 'stages': [
  ('Compute broadband energy (no FFT)', [
   ('1.', 'E_short(m)  <-  sum_n |x[n]|^2', 'ALL frequencies in one frame'),
   ('2.', 'E_long(m) <- beta_E x E_long(m-1) + (1-beta_E) x E_short(m)', 'always updates, even during events'),
  ]),
  ('Energy-ratio trigger', [
   ('3.', 'R  <-  E_short(m) / E_long(m)', 'R~1.0 steady; spikes on transients'),
   ('4.', 'if R > zeta then E[m] <- 1', 'any 6x spike triggers'),
   ('5.', 'return E[m]', 'no band selection, no persistence'),
  ]),
 ]},
'SoD': {
 'title': 'Algorithm 6: Send-on-Delta (Correa et al. 2019)',
 'color': '#263238', 'fill': '#ECEFF1',
 'stages': [
  ('Sample-by-sample comparison', [
   ('1.', 'if |x[n] - x_ref| > Delta then', 'deviation exceeds threshold'),
   ('2.', '    transmit x[n]', 'send to sink'),
   ('3.', '    x_ref <- x[n]', 'reference includes noise!'),
   ('', '  end if', ''),
   ('', '  // x_ref random-walks with noise', ''),
   ('4.', 'return transmit decision', ''),
  ]),
 ]},
'TinyML': {
 'title': 'Algorithm 7: Autoencoder Anomaly Detection (Hammad et al. 2023)',
 'color': '#006064', 'fill': '#E0F7FA',
 'stages': [
  ('Autoencoder inference', [
   ('1.', 'x_hat  <-  A_theta(x)', 'compress 128 -> 8 -> 128'),
  ]),
  ('Reconstruction error', [
   ('2.', 'e(m)  <-  ||x - x_hat||^2', 'MSE: low=noise, high=anomaly'),
  ]),
  ('Trigger', [
   ('3.', 'if e(m) > Theta_ML then E[m] <- 1', 'frozen at training time'),
   ('4.', 'return E[m]', 'cannot adapt to noise drift'),
  ]),
 ]},
}

def draw_pseudocode(pdf, algo_key):
    info = PSEUDO[algo_key]
    fig, ax = plt.subplots(figsize=(11, 7.5))
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    ax.axis('off')
    
    # Title bar
    ax.add_patch(plt.Rectangle((1, 92), 98, 6, fc=info['fill'], ec=info['color'], lw=2, zorder=2))
    ax.text(3, 95, info['title'], fontsize=14, fontweight='bold', color=info['color'], va='center', fontfamily='sans-serif')
    
    y = 88
    FONT_CODE = 'DejaVu Sans Mono'  # Monospace for all code
    SZ_CODE = 9.5
    SZ_STAGE = 11
    SZ_COMMENT = 8
    
    for stage_name, lines in info['stages']:
        ax.text(3, y, stage_name, fontsize=SZ_STAGE, fontweight='bold', color='#0D47A1', fontfamily='sans-serif')
        y -= 3.5
        
        for ln_num, code, comment in lines:
            # Line number in purple
            if ln_num:
                ax.text(4, y, ln_num, fontsize=SZ_CODE, fontweight='bold', color='#4A148C', fontfamily=FONT_CODE)
            # Code in monospace (single string, no overlap possible)
            ax.text(9, y, code, fontsize=SZ_CODE, color='#1a1a1a', fontfamily=FONT_CODE)
            # Comment right-aligned in gray italic
            if comment:
                ax.text(97, y, '// ' + comment, fontsize=SZ_COMMENT, color='#777777', style='italic',
                       ha='right', fontfamily='sans-serif')
            y -= 2.8
        y -= 1.5
    
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

def draw_method_results(pdf, name):
    r = results[name]; m = metrics[name]; t_min = frames_t/60
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                                     gridspec_kw={'height_ratios': [3, 1]})
    for g in EVENT_GROUPS:
        for ax in [ax1, ax2]:
            ax.axvspan(frames_t[g[0]]/60, frames_t[g[-1]]/60, alpha=0.18, color='green', zorder=0)
    ax1.plot(t_min, r['stat'], color=COLORS[name], lw=0.8, alpha=0.6, label='Detection statistic')
    ax1.plot(t_min, r['thresh'], 'r-', lw=2, alpha=0.8, label='Threshold')
    if name in ['TSNFA','Zhang']:
        ax1.plot(t_min, r['nf'], 'orange', lw=1.5, ls='--', alpha=0.7, label='Noise floor')
    tp_i = [f for f in range(N_FRAMES) if r['triggers'][f] and f in EVENT_ASSOC]
    fp_i = [f for f in range(N_FRAMES) if r['triggers'][f] and f not in EVENT_ASSOC]
    fn_i = [f for f in range(N_FRAMES) if not r['triggers'][f] and f in EVENT_FRAMES]
    if tp_i: ax1.scatter(frames_t[tp_i]/60, r['stat'][tp_i], color='green', s=80, marker='^', zorder=10, label=f'TP:{len(tp_i)}')
    if fp_i: ax1.scatter(frames_t[fp_i]/60, r['stat'][fp_i], color='red', s=60, marker='x', zorder=10, label=f'FP:{len(fp_i)}', linewidths=2)
    if fn_i: ax1.scatter(frames_t[fn_i]/60, r['stat'][fn_i], color='orange', s=80, marker='v', zorder=10, label=f'FN:{len(fn_i)}')
    ax1.set_ylabel('Magnitude')
    ax1.set_title(f'{name}: DR={m["DR"]:.0f}%, FP={m["FP"]}, FN={m["FN"]}, Prec={m["Prec"]:.1f}%',
                 fontsize=13, fontweight='bold', color=COLORS[name])
    ax1.legend(fontsize=8, loc='upper right', ncol=3)
    ax2.plot(t_min, 10*np.log10(frames_P/P0), 'purple', lw=1.5, alpha=0.7)
    ax2.set(ylabel='Noise (dB)', xlabel='Time (minutes)', ylim=(-8,8))
    ax2.axhline(0, color='gray', ls='--', lw=0.5)
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

print('Report functions defined (monospace pseudocode, no overlap).')

In [ ]:
# ===== Generate the PDF report =====
report_file = 'TSNFA_Simulation_Report.pdf'

with PdfPages(report_file) as pdf:
    # --- Title page ---
    fig, ax = plt.subplots(figsize=(11, 8))
    ax.axis('off')
    ax.text(0.5, 0.72, 'TSNFA Simulation Report', ha='center', fontsize=28, fontweight='bold', color='#0D47A1', transform=ax.transAxes)
    ax.text(0.5, 0.62, 'Temporal Spectral Noise-Floor Adaptation\nfor IoT Mesh Sensor Networks', ha='center', fontsize=16, color='#333', transform=ax.transAxes)
    ax.text(0.5, 0.50, 'Makovetskii & Thomsen, IEEE IoT Journal, 2026', ha='center', fontsize=12, color='#666', transform=ax.transAxes)
    ax.text(0.5, 0.38, f'{N_FRAMES} frames | {len(EVENT_GROUPS)} events | 6 algorithms compared', ha='center', fontsize=11, color='#888', transform=ax.transAxes)
    # Summary table
    table_data = [['Method', 'DR%', 'FP', 'FN', 'Prec%', 'FAR/hr']]
    for n in ['TSNFA','Zhang','STFT','DEDaR','SoD','TinyML']:
        m = metrics[n]
        table_data.append([n, f'{m["DR"]:.0f}%', str(m['FP']), str(m['FN']), f'{m["Prec"]:.1f}%', f'{m["FAR"]:.1f}'])
    tbl = ax.table(cellText=table_data[1:], colLabels=table_data[0], loc='center',
                   cellLoc='center', bbox=[0.15, 0.02, 0.7, 0.30])
    tbl.auto_set_font_size(False); tbl.set_fontsize(10)
    for (row, col), cell in tbl.get_celld().items():
        if row == 0: cell.set_facecolor('#E3F2FD'); cell.set_text_props(fontweight='bold')
        elif col == 0:
            n = table_data[row][0]
            cell.set_text_props(fontweight='bold', color=COLORS.get(n, '#333'))
    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)
    
    # --- For each method: pseudocode page + results page ---
    method_order_report = ['TSNFA', 'Zhang', 'STFT', 'DEDaR', 'SoD', 'TinyML']
    for name in method_order_report:
        if name in PSEUDO:
            draw_pseudocode(pdf, name)
        draw_method_results(pdf, name)
    
    # --- Trigger timeline ---
    fig, ax = plt.subplots(figsize=(11, 5))
    for g in EVENT_GROUPS:
        ax.axvspan(frames_t[g[0]]/60, frames_t[g[-1]]/60, alpha=0.15, color='green', zorder=0)
    for i, n in enumerate(method_order_report):
        tr = results[n]['triggers']; trig_f = np.where(tr)[0]
        for f in trig_f:
            c = 'green' if f in EVENT_ASSOC else 'red'
            ax.plot(frames_t[f]/60, i, '|', color=c, markersize=14, markeredgewidth=1.5)
        m = metrics[n]
        ax.text(-0.3, i, f'{n} (DR={m["DR"]:.0f}%, FP={m["FP"]})', ha='right', va='center',
               fontsize=9, fontweight='bold', color=COLORS[n])
    ax.set_yticks(range(len(method_order_report))); ax.set_yticklabels(['']*len(method_order_report))
    ax.set(xlabel='Time (minutes)', title='Trigger Timeline: All Methods (green=TP, red=FP)')
    ax.set_ylim(-0.5, len(method_order_report)-0.5); ax.invert_yaxis()
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)
    
    # --- Scorecard ---
    fig, axes = plt.subplots(1, 5, figsize=(14, 4))
    for ax, (title, key) in zip(axes, [('Detection Rate (%)','DR'),('False Positives','FP'),
                                         ('False Negatives','FN'),('Precision (%)','Prec'),('FAR (/hr)','FAR')]):
        vals = [metrics[n][key] for n in method_order_report]
        bars = ax.bar(range(len(method_order_report)), vals, color=[COLORS[n] for n in method_order_report], alpha=0.85)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_xticks(range(len(method_order_report)))
        ax.set_xticklabels(method_order_report, rotation=45, ha='right', fontsize=8)
        for bar, val in zip(bars, vals):
            if val > 0:
                label = f'{val:.0f}' if val==int(val) or val>10 else f'{val:.1f}'
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), label, ha='center', va='bottom', fontsize=7)
    plt.suptitle('Final Scorecard', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

print(f'Report saved: {report_file}')
print(f'Total pages: title + {len(method_order_report)} x (pseudocode + results) + timeline + scorecard')

In [ ]:
# Download the report (works in Google Colab)
try:
    from google.colab import files
    files.download(report_file)
    print('Download started!')
except ImportError:
    print(f'Not running in Colab. Report saved locally as: {report_file}')
    print(f'You can also convert this notebook to PDF with:')
    print(f'  jupyter nbconvert --to pdf TSNFA_Step_by_Step.ipynb')

## Summary

| Defence | What it does | Methods that lack it |
|---------|-------------|---------------------|
| **Band selection** (FFT) | Isolates 1-5 Hz event band | Zhang, DEDaR, SoD |
| **Persistence** (gamma_d mean) | Requires multi-frame elevation | STFT, DEDaR, SoD, TinyML |
| **Adaptive NF** (gated EMA) | Tracks +/-6 dB noise drift | STFT, SoD, TinyML |

Full 200-node, 24-hour simulation: **100% DR, 0 FP**.

- **Full simulation:** `python simulation/IOTfulltest4-withNoise4.py --nodes 200 --preset ACCURATE`
- **Dashboard:** [gnacode.github.io/IEEE-TSNFA](https://gnacode.github.io/IEEE-TSNFA/)
- **Repository:** [github.com/Gnacode/IEEE-TSNFA](https://github.com/Gnacode/IEEE-TSNFA)